In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [3]:
batch_size = 128

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 485kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.90MB/s]


In [4]:
def vae_loss(recon_x, x, mu, logvar):

    reconstruction_loss = F.binary_cross_entropy(
        recon_x,
        x,
        reduction='sum'
    )

    kl_loss = -0.5 * torch.sum(
        1 + logvar
        - mu.pow(2)
        - logvar.exp()
    )

    return reconstruction_loss + kl_loss

In [5]:
class VAE(nn.Module):

    def __init__(self, latent_dim=2):
        super().__init__()

        # Encoder
        self.fc1 = nn.Linear(784, 400)

        self.fc_mu = nn.Linear(400, latent_dim)
        self.fc_logvar = nn.Linear(400, latent_dim)

        # Decoder
        self.fc2 = nn.Linear(latent_dim, 400)
        self.fc3 = nn.Linear(400, 784)

    def encode(self, x):
        h = F.relu(self.fc1(x))

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar

    def reparameterize(self, mu, logvar):

        std = torch.exp(0.5 * logvar)

        eps = torch.randn_like(std)

        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc2(z))

        return torch.sigmoid(self.fc3(h))

    def forward(self, x):

        mu, logvar = self.encode(x)

        z = self.reparameterize(mu, logvar)

        reconstruction = self.decode(z)

        return reconstruction, mu, logvar

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = VAE(latent_dim=2).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, _ in train_loader:

        images = images.to(device)

        images = images.view(-1, 784)

        optimizer.zero_grad()

        recon, mu, logvar = model(images)

        loss = vae_loss(
            recon,
            images,
            mu,
            logvar
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f'Epoch {epoch+1}: '
        f'Loss {total_loss/len(train_loader.dataset):.4f}'
    )

Epoch 1: Loss 190.7237
Epoch 2: Loss 168.6684
Epoch 3: Loss 163.3067
Epoch 4: Loss 160.6102
Epoch 5: Loss 158.8653
Epoch 6: Loss 157.4836
Epoch 7: Loss 156.4293
Epoch 8: Loss 155.5461
Epoch 9: Loss 154.7971
Epoch 10: Loss 154.1574


In [7]:
model.eval()

with torch.no_grad():

    z = torch.randn(16, 2).to(device)

    generated = model.decode(z)

    generated = generated.view(-1, 28, 28)